**Agenda**

Um agente de LLM nunca responde só a partir da mensagem que chega. Cada resposta é montada com pedaços de
informação que vêm de lugares diferentes, duram tempos diferentes e são lidos de jeitos diferentes.

- estado, configuração, memória e conhecimento: os quatro tipos de contexto;
- onde cada um vive na arquitetura do agente;
- Context Engineering como a decisão de o que entra no prompt.

# [Conceito] Gerenciando Contexto

## Atendimento em um plantão

Pense numa pessoa da recepção da Clínica Alura no meio de um plantão, atendendo pacientes um atrás do outro. Cada
tipo de contexto corresponde a uma fonte de informação diferente que ela usa para fazer isso.

Config e Conhecimento alimentam os nós do grafo, o Estado nasce e morre dentro de cada execução, e o Output
de uma execução pode ser adicionado a memória, e compõe na entrada da próxima.

<img src="resources/context_types_overview.png" width="80%">

## Estado: o que existe durante esta execução

O estado é a informação que o grafo carrega enquanto está rodando: as mensagens da conversa, um campo
`intent` calculado por um nó, uma lista de notas acumuladas por um reducer. Cada nó lê uma parte do estado e
devolve uma atualização.

O estado nasce no início de uma execução e desaparece no fim dela, a não ser que alguma coisa o persista de
propósito. É o bloco de notas que a recepcionista rabisca durante um atendimento: existe enquanto o
atendimento dura, e some junto com ele.

## Configuração: o que entra pronto na invocação

Nem toda informação precisa nascer dentro do grafo. Um `patient_id`, a unidade de atendimento, uma flag de
ambiente: são valores que já existem antes da execução começar e não mudam durante ela. Esse tipo de
contexto entra pronto, na hora de chamar o grafo (`invoke`), e fica disponível para leitura em qualquer nó ou
tool, sem precisar passar por parâmetro em cada função. Quem decide até quando um valor continua valendo é
quem chama o grafo, não o grafo em si: o mesmo `patient_id` pode se repetir em muitas execuções seguidas,
enquanto aquele paciente estiver identificado.

O que diferencia configuração de estado é quem escreve nela, não por quanto tempo ela vale: configuração é
só lida, nunca escrita pelo grafo. É o crachá que a recepcionista recebe ao bater o ponto no início do
plantão: vale para todos os atendimentos daquele turno, não só para um paciente.

## Memória de curto prazo: o que sobrevive entre execuções

Um grafo comum começa do zero a cada `invoke`. Sem nada guardando o que aconteceu antes, a segunda mensagem
de uma conversa não sabe da primeira. Memória de curto prazo é o mecanismo que resolve isso: um
`checkpointer` salva o estado de uma execução e recupera na próxima, desde que as duas compartilhem o mesmo
`thread_id`, o identificador de conversa do LangGraph.

O escopo é esse `thread_id`: duas conversas com `thread_id`s diferentes não se enxergam, mesmo rodando o
mesmo grafo. É a pasta do paciente guardada na gaveta, reaberta só quando ele volta, e só se for o mesmo
paciente.

<img src="resources/thread_id_scope.png" width="100%">

## Conhecimento: o que vive fora do modelo

Um modelo de linguagem não sabe o nome dos pacientes da Clínica Alura, nem os horários de funcionamento
reais. Esse tipo de dado vive fora do modelo, numa fonte estruturada, e chega ao agente através de uma tool
que consulta essa fonte sob demanda.

A diferença real para os outros três é essa: configuração chega pronta no `invoke`, mas conhecimento não.
Uma tool precisa buscá-lo ativamente, sempre que for necessário. É o manual de políticas na estante: a
recepcionista não decorou o conteúdo, só consulta quando precisa.

## Os quatro tipos, lado a lado

| Tipo | Onde vive | Como o agente acessa | Quanto dura |
|---|---|---|---|
| Estado | Dentro do grafo, escopo de uma execução | `state`, `Command` para escrever | Uma execução |
| Configuração | Fora do grafo, definida por quem chama a cada `invoke` | `context_schema` + `ToolRuntime`/`Runtime` | Quantas execuções o chamador repetir o mesmo valor |
| Memória (curto prazo) | Fora do grafo, escopo de um `thread_id` | `checkpointer` + `thread_id` | Várias execuções do mesmo `thread_id` |
| Conhecimento | Fora do agente, numa fonte de dados compartilhada | Uma tool que consulta a fonte | Indefinidamente, até o dado mudar |

## Uma nota sobre nomenclatura

A documentação oficial da LangChain reconhece três fontes de contexto: configuração de runtime, estado e um
armazenamento de longo prazo (store, fora do escopo deste curso). Ela trata estado e memória de curto prazo
como o mesmo dado, o estado persistido por um checkpointer, não dois conceitos separados. Aqui separamos os
dois momentos, execução única e várias execuções, para focar em um de cada vez. Conhecimento, o quarto tipo
desta aula, é um rótulo próprio do curso, para dar lugar ao acesso determinístico a dados sem entrar em
memória de longo prazo.

## Context Engineering

Nenhum desses quatro tipos chega ao modelo automaticamente. Alguém decide, para cada um, se ele entra direto
no prompt, se vira uma chamada de tool, se filtra uma consulta, ou se nem chega perto do modelo. Essa
decisão, o que entra no prompt, quando e como, é o que se chama de Context Engineering: escolher o que fazer
com cada tipo de contexto em cada passo da execução.

## Nesta aula

1. Injetando configs e runtime context.
2. Tool Runtime: salvando e recuperando contexto.
3. Checkpoints e Short-Term Memory.
4. Projeto: dados da clínica com SQLite.